# Trading Bot V4.1 · Kraken Paper Trading mit Cockpit

V4.1 handelt mit **denselben Strategie- und Risikoregeln wie V4.0**. Die Strategie ist eine Trendfolge mit MA20/50/200, ADX ≥ 20 und ATR-Positionsgröße. Ausgestiegen wird ausschließlich über den ATR-Trailing-Stop. Gehandelt werden BTC, ETH und SOL gegen EUR, und zwar nur virtuell: **Es gibt keine echten Orders.** Das Paper-Konto `MyDrive/Trading_Bot/konto_v3_trendfolge.json` läuft unverändert weiter.

## Neu: Cockpit

Das Cockpit hat eine **eigene Zelle** („Cockpit“) und wird von der Startzelle nach jeder Kursprüfung (alle 5 Minuten) an derselben Stelle aktualisiert. Es zeigt:
- **Kacheln:** Gesamtvermögen, Gewinn/Verlust seit Start und heute (mit Tagesverlustgrenze), freies Guthaben, Marktwert der Positionen, offenes Risiko bis Stop (Balken gegen die 3-%-Grenze), realisiertes Ergebnis, Gebühren und die Anzahl der Positionen und Trades.
- **Offene Positionen:** Einstieg, aktueller Kurs, Abstand zum Stop, Ergebnis nach simulierten Kosten, Trailing-Stop und höchster Schlusskurs. Liegt der Stop unter dem Einstieg, ist er rot, darüber grün. Daneben steht, welcher Betrag nach Kosten bereits gesichert ist oder welches Risiko bis zum Stop noch besteht.
- **Letzte Marktanalyse (stündlich):** Kurs, MA20/50/200, ADX, ATR, Signal und die Entscheidung mit Begründung, auch eine Ablehnung durch das Risk Gate und eine laufende 24-h-Handelspause.
- **Abgeschlossene Trades:** die letzten 12, jeweils mit Kauf- und Verkaufszeit, Kursen, Ergebnis und Ausstiegsgrund.
- **Letzte Ereignisse:** Käufe, Verkäufe, Stop-Nachzüge, abgelehnte Käufe, API- und Datenfehler sowie das Tageslimit.

**Vollbild:** Weil die Cockpit-Zelle nur das Cockpit ausgibt, lässt sich im Ausgabemenü dieser Zelle „Copy link to fullscreen output“ wählen. Der Link zeigt dann nur das Cockpit, ohne Code und ohne Textlog. Ob diese Vollbildansicht die laufenden Updates mitzeigt oder einen festen Stand, hängt von Colab ab. Für eine sicher aktuelle Ansicht ist der Browser-Link gedacht.

**Im Browser:** Die Cockpit-Zelle zeigt den Link „Cockpit im Browser öffnen“. Die Seite lädt alle 15 s neu und zeigt, wie alt der Stand ist. Kommt länger als 11 Minuten kein neuer Stand, obwohl der Bot laufen sollte, erscheint ein roter Hinweis. Der Link funktioniert nur, solange die Colab-Sitzung läuft. Der kleine Webserver liefert ausschließlich die fertige Ansicht aus: keine Marktdaten, keine Kontoänderung, kein Dateizugriff.

**Als Datei:** `MyDrive/Trading_Bot/cockpit_latest.html` enthält immer den neuesten Stand und lädt sich im Browser ebenfalls selbst neu. Mit „Google Drive für Desktop“ lässt sie sich lokal öffnen, auch nachdem Colab beendet ist. Die Drive-Weboberfläche zeigt HTML nicht als Seite an.

Das Cockpit ist reine Anzeige. Es zeigt nur, was der Bot im Durchlauf ohnehin berechnet hat, und fragt keine Kurse zusätzlich ab. Tritt im Cockpit ein Fehler auf, schaltet es sich ab und der Handel läuft unverändert weiter. Die farbige Textausgabe von V4.0 bleibt vollständig erhalten und erscheint wie bisher unter der Startzelle.

## Korrekturen gegenüber V4.0

1. **Trailing-Stop über dem Einstieg:** V4.0 hat beim Speichern `stop_loss < einstieg` verlangt. Sobald der Trailing-Stop in einem guten Trend über den Einstieg stieg, brach der Bot mit „Ungültige Kursmarken“ ab. Diese Prüfung ist entfallen, und ein solcher Stop zählt mit 0 € Risiko.
2. **Höchster Schlusskurs:** Er wird jetzt auch gespeichert, wenn der Stop dabei nicht steigt. In V4.0 ging er verloren, weil das Konto in jedem Durchlauf neu aus der Datei geladen wird.

Alle anderen Regeln sind unverändert: Einstieg, Positionsgröße, Risk Gate, Stop-Prüfung alle 5 Minuten, Tagesverlustlimit und Handelspause.

## Aufbau

Die beiden `%%writefile`-Zellen schreiben `crypto_bot_v41.py` (Bot) und `crypto_cockpit_v41.py` (Cockpit). Danach werden der Reihe nach die Cockpit-Zelle und die Startzelle ausgeführt (oder einfach „Alle ausführen“). Die Tests im Repository (`pytest`) laufen gegen genau diese Zellen.

Ergebnisse im Paper Trading sind Modellwerte und kein Rentabilitätsnachweis.

In [ ]:
%%writefile /content/crypto_bot_v41.py
"""Trading Bot V4.1: Kraken Paper Trading, Trendfolge. Keine echten Orders."""
# ============================================================
#
# TRADING BOT V4.1
#
# KRAKEN + PAPER TRADING + RISIKOMANAGEMENT
# TRENDFOLGE: MA20/50/200 + ADX-Stärkefilter
# ATR-Positionsgröße + ATR-Trailing-Stop (kein festes Take-Profit)
#
# BTC / ETH / SOL
#
# Kontowährung: EUR
#
# Neu in V4.1:
# - Cockpit (Colab-Ansicht, Browser-Link, cockpit_latest.html)
#   über crypto_cockpit_v41.py - reine Anzeige, keine Kursabfragen
# - Trailing-Stop über dem Einstieg bricht den Bot nicht mehr ab
# - höchster Schlusskurs wird immer gespeichert
#
# Das Modul hat beim Import keine Seiteneffekte (kein Drive,
# keine Kraken-Abfrage) - gestartet wird über start().
#
# ============================================================


# ============================================================
# 1. BIBLIOTHEKEN
# ============================================================

import requests
import pandas as pd

import json
import time
import math
import os
import re
import shutil
import traceback

from collections import deque
from pathlib import Path
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo


# ------------------------------------------------------------
# AUSGABE-DESIGN (Farben, Kästen, Emojis für Google Colab)
# ------------------------------------------------------------

FARBE_GRUEN = "\033[92m"
FARBE_ROT = "\033[91m"
FARBE_GELB = "\033[93m"
FARBE_BLAU = "\033[94m"
FARBE_CYAN = "\033[96m"
FARBE_GRAU = "\033[90m"
FARBE_FETT = "\033[1m"
FARBE_ENDE = "\033[0m"

BREITE = 60


def faerben(text, farbe):

    return f"{farbe}{text}{FARBE_ENDE}"


def wert_faerben(zahl, text):

    # Grün bei >= 0, Rot bei < 0 - für Gewinn/Verlust-Angaben

    return faerben(
        text,
        FARBE_GRUEN if zahl >= 0 else FARBE_ROT
    )


def kasten(titel, emoji=""):

    linie = "═" * BREITE

    print("")

    print(faerben(linie, FARBE_BLAU))

    kopf = f"{emoji}  {titel}" if emoji else titel

    print(faerben(kopf.center(BREITE), FARBE_FETT))

    print(faerben(linie, FARBE_BLAU))


def unterkasten(titel, emoji=""):

    kopf = f"{emoji}  {titel}" if emoji else titel

    print("")

    print(faerben(kopf, FARBE_FETT))

    print(faerben("─" * BREITE, FARBE_GRAU))


def trenner():

    print(faerben("─" * BREITE, FARBE_GRAU))


def zeile(label, wert, emoji=None, breite_label=26):

    praefix = f"{emoji} " if emoji else ""

    print(
        f"{praefix}{label:<{breite_label}} {wert}"
    )


# --- Dashboard-Boxen (┌─┐ │ └─┘) für Portfolio/Positionen/Signale ---

INNEN_BREITE = 62

# Emojis/Sonderzeichen wirken in Monospace-Schriften (Colab)
# doppelt so breit wie ihre Länge in Python (len() zählt 1
# Zeichen). Für einigermaßen saubere Rahmen wird das hier grob
# nachgebildet - perfekte Pixel-Ausrichtung ist nicht das Ziel.
# ANSI-Farbcodes sind unsichtbar und zählen nicht zur Breite.

ANSI_MUSTER = re.compile(r"\033\[[0-9;]*m")


def sichtbare_breite(text):

    text_ohne_farbcodes = ANSI_MUSTER.sub("", text)

    breite = 0

    for zeichen in text_ohne_farbcodes:

        code = ord(zeichen)

        if code == 0xFE0F:

            continue

        if (
            0x1F300 <= code <= 0x1FAFF
            or 0x2600 <= code <= 0x27BF
            or code in (0x25B2, 0x25BC, 0x25CF)
        ):

            breite += 2

        else:

            breite += 1

    return breite


def box_start(titel, emoji=""):

    kopf_text = f"{emoji} {titel}" if emoji else titel

    kopf = f"┌─ {kopf_text} "

    # Gesamtbreite der oberen Rahmenlinie muss der Breite von
    # box_zeile()/box_ende() entsprechen: "│ " + INNEN_BREITE +
    # " │" bzw. "└" + "─"*(INNEN_BREITE+2) + "┘" = INNEN_BREITE+4

    rest = (
        INNEN_BREITE + 4
        - sichtbare_breite(kopf)
        - 1
    )

    if rest < 0:

        rest = 0

    print("")

    print(
        faerben(kopf, FARBE_BLAU)
        + faerben("─" * rest + "┐", FARBE_BLAU)
    )


def box_zeile(text=""):

    innen = sichtbare_breite(text)

    fuellung = max(
        0,
        INNEN_BREITE - innen
    )

    print(
        faerben("│ ", FARBE_BLAU)
        + text
        + " " * fuellung
        + faerben(" │", FARBE_BLAU)
    )


def spalte(text, breite):

    # ANSI-sicheres Auffüllen einer Tabellenspalte auf feste
    # sichtbare Breite - normales str.ljust() würde die
    # unsichtbaren Farbcodes mitzählen und die Spalte dadurch
    # zu schmal auffüllen.

    fehlend = max(
        0,
        breite - sichtbare_breite(text)
    )

    return text + " " * fehlend


def box_ende():

    print(
        faerben(
            "└" + "─" * (INNEN_BREITE + 2) + "┘",
            FARBE_BLAU
        )
    )


def balken(anteil, breite=20):

    anteil = max(0.0, min(1.0, anteil))

    gefuellt = round(anteil * breite)

    farbe = FARBE_GRUEN if anteil < 0.8 else FARBE_ROT

    fuellstrich = faerben("█" * gefuellt, farbe)

    leerstrich = faerben(
        "░" * (breite - gefuellt),
        FARBE_GRAU
    )

    return f"[{fuellstrich}{leerstrich}]"


def pfeil(wert):

    if wert > 0:

        return faerben("▲", FARBE_GRUEN)

    if wert < 0:

        return faerben("▼", FARBE_ROT)

    return faerben("─", FARBE_GRAU)


def signal_text(signal):

    if signal == "LONG":

        return faerben("🟢 LONG", FARBE_GRUEN)

    return faerben("⚪ NO TRADE", FARBE_GRAU)


VERSION = "4.1"


# ============================================================
# 2. GOOGLE DRIVE VERBINDEN
# ============================================================

# Passiert erst in start() - siehe Abschnitt 27.


# ============================================================
# 3. GRUNDEINSTELLUNGEN
# ============================================================

STARTKAPITAL = 10000.00

# Ausschließlich Paper Trading
PAPER_TRADING = True

# Automatische virtuelle Käufe
AUTOMATISCHE_KAEUFE = True

# Kursüberwachung alle 5 Minuten
INTERVALL = 300

# Risikomanagement
MAX_RISIKO_PRO_TRADE = 0.01
MAX_GESAMTRISIKO = 0.03
MAX_POSITIONSANTEIL = 0.50

MAX_OFFENE_POSITIONEN = 3

MAX_TAGESVERLUST = 0.03

# Handelsstrategie: Trendfolge mit ATR-Positionsgröße
# und ATR-Trailing-Stop (kein fixes Take-Profit, siehe Analyse
# zur Kostenasymmetrie: fixe kleine Targets verlieren gegen
# Gebühren+Slippage, laufengelassene Trends tragen sie)

TREND_FILTER_PERIODE = 200      # langsamer Trendfilter (MA200)

ADX_PERIODE = 14
ADX_MINDESTWERT = 20            # Mindest-Trendstärke für ein Signal

ATR_PERIODE = 14
ATR_STOP_MULTIPLIKATOR = 2.0        # initialer Stop = Einstieg - 2x ATR
ATR_TRAILING_MULTIPLIKATOR = 3.0    # Trailing-Stop = höchster Schluss seit Einstieg - 3x ATR

# Mindestanzahl an Stundenkerzen, bevor gehandelt wird
# (Platz für MA200 + Warmlaufzeit von ATR/ADX)
MIN_ERFORDERLICHE_KERZEN = TREND_FILTER_PERIODE + 50

# Simulierte Handelskosten (Kraken: Taker ~0,26%, hier bewusst
# konservativer angesetzt inkl. Slippage-Puffer)
GEBUEHR = 0.001
SLIPPAGE = 0.002

# Handelspause nach Verkauf
HANDELSPAUSE = 86400

# Zeitintervall der historischen Kerzen
KERZEN_INTERVALL = 60

# Sekunden pro Stundenkerze
KERZEN_SEKUNDEN = 3600


# ============================================================
# 4. KRYPTOWÄHRUNGEN
# ============================================================

COINS = {

    "bitcoin": {
        "name": "Bitcoin",
        "paar": "XBT/EUR"
    },

    "ethereum": {
        "name": "Ethereum",
        "paar": "ETH/EUR"
    },

    "solana": {
        "name": "Solana",
        "paar": "SOL/EUR"
    }

}


# ============================================================
# 5. SPEICHERORTE
# ============================================================

STANDARD_ORDNER = Path(
    "/content/drive/MyDrive/Trading_Bot"
)

# Werden von speicherorte_einrichten() gesetzt (in Tests auf
# einen temporären Ordner statt Google Drive).

ORDNER = None
MARKTDATEN_ORDNER = None
KONTO_DATEI = None
BACKUP_DATEI = None
ANALYSE_DATEI = None


def speicherorte_einrichten(ordner=STANDARD_ORDNER):

    global ORDNER, MARKTDATEN_ORDNER, KONTO_DATEI
    global BACKUP_DATEI, ANALYSE_DATEI

    ORDNER = Path(ordner)

    ORDNER.mkdir(
        parents=True,
        exist_ok=True
    )

    MARKTDATEN_ORDNER = ORDNER / "Marktdaten_Kraken"

    MARKTDATEN_ORDNER.mkdir(
        parents=True,
        exist_ok=True
    )

    # Dieselbe Kontodatei wie V4.0 - das Datenschema ist
    # unverändert, das Paper-Konto läuft also einfach weiter.
    # (Die alte konto_v2.json aus V3.1 bleibt unangetastet.)

    KONTO_DATEI = ORDNER / "konto_v3_trendfolge.json"

    # Sicherung der bisherigen Kontodatei

    BACKUP_DATEI = ORDNER / "konto_v3_trendfolge_backup.json"

    # Ergebnis der letzten stündlichen Marktanalyse - nur für
    # die Anzeige, damit das Cockpit nach einem Neustart
    # innerhalb derselben Stunde nicht leer ist.

    ANALYSE_DATEI = ORDNER / "letzte_marktanalyse.json"


# ============================================================
# 6. HILFSFUNKTIONEN
# ============================================================

def zeitstempel():

    # Intern wird konsequent UTC gespeichert (Kontodatei,
    # Transaktionen, Kraken-Stundenkerzen) - das hält Tages-
    # limit-Reset, 24h-Handelspause und Kerzengrenzen robust
    # gegen die Zeitumstellung. Nur die Anzeige für den
    # Menschen läuft über lokale_zeit_anzeigen() unten.

    return datetime.now(
        timezone.utc
    ).isoformat()


BERLIN_ZEITZONE = ZoneInfo("Europe/Berlin")


def lokale_zeit_anzeigen(zeitpunkt_utc):

    # Wandelt einen UTC-Zeitpunkt für die Anzeige nach
    # Europe/Berlin um - mit dem korrekten deutschen Kürzel
    # (MESZ im Sommer, MEZ im Winter). datetime.strftime("%Z")
    # liefert dafür nur "CEST"/"CET" (englisch), daher hier
    # manuell über die Sommerzeit-Erkennung gesetzt.

    lokal = zeitpunkt_utc.astimezone(BERLIN_ZEITZONE)

    ist_sommerzeit = (
        lokal.dst() is not None
        and lokal.dst().total_seconds() > 0
    )

    kuerzel = "MESZ" if ist_sommerzeit else "MEZ"

    return lokal.strftime(f"%d.%m.%Y %H:%M:%S {kuerzel}")


def euro(wert):

    text = f"{wert:,.2f}"

    text = (
        text.replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    return text + " €"


def gueltige_zahl(wert):

    return (
        isinstance(wert, (int, float))
        and not isinstance(wert, bool)
        and math.isfinite(wert)
    )


# Ereignisprotokoll für das Cockpit (nur im Speicher, geht bei
# einem Neustart verloren - Käufe/Verkäufe stehen dauerhaft in
# den Transaktionen der Kontodatei).

EREIGNISSE = deque(maxlen=50)


def ereignis(art, coin=None, text=""):

    EREIGNISSE.append({
        "zeit": zeitstempel(),
        "art": art,
        "coin": coin,
        "text": text
    })


# ============================================================
# 7. KRAKEN API
# ============================================================

KRAKEN_API = "https://api.kraken.com/0/public"

session = requests.Session()

# Kurze Pause zwischen API-Anfragen
LETZTE_API_ANFRAGE = 0


def kraken_anfrage(endpunkt, parameter=None):

    global LETZTE_API_ANFRAGE

    url = f"{KRAKEN_API}/{endpunkt}"

    for versuch in range(3):

        # Mindestens 1,5 Sekunden Abstand
        # zwischen zwei Kraken-Anfragen

        abstand = time.monotonic() - LETZTE_API_ANFRAGE

        if abstand < 1.5:

            time.sleep(1.5 - abstand)

        LETZTE_API_ANFRAGE = time.monotonic()

        try:

            response = session.get(
                url,
                params=parameter,
                timeout=20
            )

            # HTTP-Limit erreicht

            if response.status_code == 429:

                wartezeit = 30 * (versuch + 1)

                print(
                    f"Kraken API-Limit erreicht. "
                    f"Warte {wartezeit} Sekunden."
                )

                time.sleep(wartezeit)

                continue

            response.raise_for_status()

            daten = response.json()

            # Kraken meldet API-Fehler im JSON

            fehler = daten.get("error", [])

            if fehler:

                fehlermeldung = ", ".join(fehler)

                # Vorübergehende API-Probleme

                if any(
                    "Rate limit" in f
                    or "Service:Unavailable" in f
                    or "Temporary" in f
                    for f in fehler
                ):

                    time.sleep(30 * (versuch + 1))

                    continue

                raise RuntimeError(
                    f"Kraken API: {fehlermeldung}"
                )

            return daten["result"]

        except (
            requests.exceptions.Timeout,
            requests.exceptions.ConnectionError
        ):

            if versuch == 2:

                raise

            time.sleep(10 * (versuch + 1))

    raise RuntimeError(
        f"Kraken-Anfrage fehlgeschlagen: {endpunkt}"
    )


# ============================================================
# 8. KRAKEN-HANDELSPAARE ERMITTELN
# ============================================================

def handelspaare_ermitteln():

    print("")
    print("Prüfe verfügbare Kraken-Handelspaare ...")

    daten = kraken_anfrage("AssetPairs")

    ergebnis = {}

    for coin, info in COINS.items():

        gesucht = info["paar"]

        treffer = []

        for kennung, paar_info in daten.items():

            if paar_info.get("wsname") == gesucht:

                treffer.append(
                    (kennung, paar_info)
                )

        # Zusätzliche Märkte und Spezialpaare
        # dürfen nicht versehentlich ausgewählt werden.

        treffer = [
            (kennung, info)
            for kennung, info in treffer
            if ".d" not in kennung.lower()
        ]

        if len(treffer) != 1:

            raise RuntimeError(
                f"Handelspaar nicht eindeutig: {gesucht}"
            )

        kennung, paar_info = treffer[0]

        ergebnis[coin] = kennung

        print(
            f"{info['name']}: "
            f"{gesucht} ({kennung})"
        )

    return ergebnis


# Wird in start() über handelspaare_ermitteln() gesetzt
HANDELSPAARE = {}


# ============================================================
# 9. AKTUELLE KURSE ABRUFEN
# ============================================================

def aktuelle_kurse():

    parameter = {
        "pair": ",".join(HANDELSPAARE.values())
    }

    daten = kraken_anfrage(
        "Ticker",
        parameter
    )

    kurse = {}

    for coin, kennung in HANDELSPAARE.items():

        if kennung not in daten:

            raise RuntimeError(
                f"Ticker fehlt: {coin}"
            )

        # Kraken liefert:
        # a = aktueller Verkaufspreis / Ask
        # b = aktueller Kaufpreis / Bid
        # c = letzter gehandelter Preis

        kurs = float(
            daten[kennung]["c"][0]
        )

        if not gueltige_zahl(kurs) or kurs <= 0:

            raise ValueError(
                f"Ungültiger Kraken-Kurs: {coin}"
            )

        kurse[coin] = kurs

    return kurse


# ============================================================
# 10. HISTORISCHE KURSE SPEICHERN
# ============================================================

def kursdatei(coin):

    return (
        MARKTDATEN_ORDNER
        / f"{coin}_1h.csv"
    )


def kurse_laden(coin):

    datei = kursdatei(coin)

    if not datei.exists():

        return pd.DataFrame(
            columns=["open", "high", "low", "close", "volume"],
            index=pd.DatetimeIndex([], tz="UTC", name="zeit")
        )

    df = pd.read_csv(
        datei,
        parse_dates=["zeit"],
        index_col="zeit"
    )

    df.index = pd.to_datetime(
        df.index,
        utc=True
    )

    if df.index.has_duplicates:

        raise ValueError(
            f"Doppelte Zeitstempel in {datei}"
        )

    df = df.sort_index()

    return df


def kurse_speichern(coin, df):

    datei = kursdatei(coin)

    temporaer = Path(
        str(datei) + ".tmp"
    )

    df.to_csv(
        temporaer,
        index_label="zeit"
    )

    os.replace(
        temporaer,
        datei
    )


# ============================================================
# 11. HISTORISCHE KURSE AKTUALISIEREN
# ============================================================

def historische_kurse(coin):

    df_alt = kurse_laden(coin)

    parameter = {

        "pair": HANDELSPAARE[coin],

        "interval": KERZEN_INTERVALL

    }

    # Bereits gespeicherte Daten nicht erneut
    # vollständig herunterladen.

    if not df_alt.empty:

        letzter_zeitpunkt = int(
            df_alt.index[-1].timestamp()
        )

        # Letzte bekannte Kerze nochmals abrufen,
        # damit Übergänge überprüft werden können.

        parameter["since"] = (
            letzter_zeitpunkt - KERZEN_SEKUNDEN
        )

    daten = kraken_anfrage(
        "OHLC",
        parameter
    )

    kennung = HANDELSPAARE[coin]

    if kennung not in daten:

        raise RuntimeError(
            f"OHLC-Daten fehlen: {coin}"
        )

    kerzen = daten[kennung]

    zeilen = []

    aktuelle_stunde = (
        pd.Timestamp.now(tz="UTC").floor("h")
    )

    for kerze in kerzen:

        zeit = pd.to_datetime(
            int(kerze[0]),
            unit="s",
            utc=True
        )

        # Noch laufende Stundenkerze ausschließen

        if zeit >= aktuelle_stunde:

            continue

        zeilen.append({

            "zeit": zeit,

            "open": float(kerze[1]),

            "high": float(kerze[2]),

            "low": float(kerze[3]),

            "close": float(kerze[4]),

            "volume": float(kerze[6])

        })

    if not zeilen:

        raise RuntimeError(
            f"Keine abgeschlossenen Kerzen: {coin}"
        )

    df_neu = pd.DataFrame(
        zeilen
    ).set_index("zeit")

    # Bereits gespeicherte und neue Kurse
    # zusammenführen.

    df = pd.concat([
        df_alt,
        df_neu
    ])

    df = df[
        ~df.index.duplicated(keep="last")
    ]

    df = df.sort_index()

    # Nur vollständige Stunden speichern

    df = df[
        df.index < aktuelle_stunde
    ]

    # Daten plausibilisieren

    for spalte in [
        "open",
        "high",
        "low",
        "close"
    ]:

        if (
            df[spalte].isna().any()
            or (df[spalte] <= 0).any()
        ):

            raise ValueError(
                f"Ungültige historische Kurse: {coin}"
            )

    # Neue Daten dauerhaft speichern

    kurse_speichern(
        coin,
        df
    )

    print(
        f"{COINS[coin]['name']}: "
        f"{len(df)} Stundenkerzen gespeichert."
    )

    return df


# ============================================================
# 12. TECHNISCHE INDIKATOREN (ATR, ADX)
# ============================================================

def true_range_berechnen(df):

    vorheriger_schluss = df["close"].shift(1)

    bereich = pd.concat([

        df["high"] - df["low"],

        (df["high"] - vorheriger_schluss).abs(),

        (df["low"] - vorheriger_schluss).abs()

    ], axis=1).max(axis=1)

    return bereich


def atr_berechnen(df, periode=ATR_PERIODE):

    tr = true_range_berechnen(df)

    atr = tr.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    return atr


def adx_berechnen(df, periode=ADX_PERIODE):

    hoch_diff = df["high"].diff()

    tief_diff = -df["low"].diff()

    plus_dm = hoch_diff.where(
        (hoch_diff > tief_diff) & (hoch_diff > 0),
        0.0
    )

    minus_dm = tief_diff.where(
        (tief_diff > hoch_diff) & (tief_diff > 0),
        0.0
    )

    tr = true_range_berechnen(df)

    tr_geglaettet = tr.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    plus_dm_geglaettet = plus_dm.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    minus_dm_geglaettet = minus_dm.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    plus_di = 100 * (
        plus_dm_geglaettet / tr_geglaettet
    )

    minus_di = 100 * (
        minus_dm_geglaettet / tr_geglaettet
    )

    dx = 100 * (
        (plus_di - minus_di).abs()
        / (plus_di + minus_di)
    )

    adx = dx.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    return adx


# ============================================================
# 13. MARKTDATEN VALIDIEREN
# ============================================================

def marktdaten_pruefen(df, coin):

    # Genug historische Kerzen für MA200 + Indikator-Warmlauf

    if len(df) < MIN_ERFORDERLICHE_KERZEN:

        raise ValueError(
            f"Zu wenige historische Kurse: {coin}"
        )

    # Die letzten 50 Stunden müssen lückenlos sein.

    letzte_kerzen = df.tail(50)

    abstaende = (
        letzte_kerzen.index.to_series()
        .diff()
        .dropna()
    )

    if not (
        abstaende == pd.Timedelta(hours=1)
    ).all():

        raise ValueError(
            f"Lücke in den historischen Kursdaten: {coin}"
        )

    # Keine Handelssignale aus veralteten Daten

    aktuelle_stunde = (
        pd.Timestamp.now(tz="UTC").floor("h")
    )

    letzter_zeitpunkt = df.index[-1]

    if letzter_zeitpunkt < (
        aktuelle_stunde - pd.Timedelta(hours=1)
    ):

        raise ValueError(
            f"Historische Kurse veraltet: {coin}"
        )

    return True


# ============================================================
# 14. HANDELSKONTO
# ============================================================

def neues_konto():

    return {

        "startkapital": STARTKAPITAL,

        "guthaben": STARTKAPITAL,

        "positionen": {},

        "transaktionen": [],

        "erstellt_am": zeitstempel(),

        "tagesdatum": None,

        "tagesstartwert": STARTKAPITAL,

        "tageslimit_erreicht": False,

        "letzte_analyse": None,

        "letzter_ausstieg": {}

    }


def konto_pruefen(konto):

    if not isinstance(konto, dict):

        raise ValueError(
            "Ungültige Kontodatei."
        )

    for feld in [
        "startkapital",
        "guthaben",
        "positionen",
        "transaktionen"
    ]:

        if feld not in konto:

            raise ValueError(
                f"Kontofeld fehlt: {feld}"
            )

    if (
        not gueltige_zahl(konto["guthaben"])
        or konto["guthaben"] < -0.000001
    ):

        raise ValueError(
            "Ungültiges Kontoguthaben."
        )

    if not isinstance(
        konto["positionen"], dict
    ):

        raise ValueError(
            "Ungültige Positionsdaten."
        )

    if not isinstance(
        konto["transaktionen"], list
    ):

        raise ValueError(
            "Ungültige Transaktionshistorie."
        )

    for coin, position in konto["positionen"].items():

        if coin not in COINS:

            raise ValueError(
                f"Unbekannte Kryptowährung: {coin}"
            )

        for feld in [
            "menge",
            "einstieg",
            "stop_loss",
            "gesamtkosten",
            "hoechster_schlusskurs",
            "atr_bei_einstieg"
        ]:

            wert = position.get(feld)

            if (
                not gueltige_zahl(wert)
                or wert <= 0
            ):

                raise ValueError(
                    f"Ungültige Position: {coin}, {feld}"
                )

        # Keine Prüfung "stop_loss < einstieg" mehr (V4.0):
        # Der Trailing-Stop steigt in einem guten Trend
        # planmäßig über den Einstieg. V4.0 hat dann beim
        # Speichern einen Fehler geworfen und sich selbst
        # gestoppt. offenes_risiko_berechnen() rechnet einen
        # solchen Stop bereits mit 0 € Risiko.


def konto_speichern(konto):

    konto_pruefen(konto)

    temporaer = Path(
        str(KONTO_DATEI) + ".tmp"
    )

    with open(
        temporaer,
        "w"
    ) as datei:

        json.dump(
            konto,
            datei,
            indent=4,
            ensure_ascii=False
        )

    os.replace(
        temporaer,
        KONTO_DATEI
    )


def konto_laden():

    if KONTO_DATEI.exists():

        with open(
            KONTO_DATEI,
            "r"
        ) as datei:

            konto = json.load(datei)

        konto_pruefen(konto)

        return konto

    konto = neues_konto()

    konto_speichern(konto)

    return konto


# ============================================================
# 15. BESTEHENDES KONTO ÜBERNEHMEN
# ============================================================

def konto_uebernehmen():

    if (
        KONTO_DATEI.exists()
        and not BACKUP_DATEI.exists()
    ):

        shutil.copy2(
            KONTO_DATEI,
            BACKUP_DATEI
        )

        print(
            "Sicherung des bisherigen Kontos erstellt."
        )

    konto = konto_laden()

    konto.setdefault("tagesdatum", None)

    konto.setdefault(
        "tagesstartwert",
        STARTKAPITAL
    )

    konto.setdefault(
        "tageslimit_erreicht",
        False
    )

    konto.setdefault(
        "letzte_analyse",
        None
    )

    konto.setdefault(
        "letzter_ausstieg",
        {}
    )

    # Verkaufszeitpunkte aus der bisherigen
    # Transaktionshistorie übernehmen.

    for transaktion in konto["transaktionen"]:

        if transaktion.get("aktion") == "VERKAUF":

            coin = transaktion.get("coin")

            verkaufszeit = transaktion.get("zeit")

            if coin in COINS and verkaufszeit:

                bisher = konto["letzter_ausstieg"].get(coin)

                if (
                    bisher is None
                    or verkaufszeit > bisher
                ):

                    konto["letzter_ausstieg"][coin] = (
                        verkaufszeit
                    )

    konto_speichern(konto)

    unterkasten("Handelskonto geladen", "💾")

    zeile("Guthaben", euro(konto["guthaben"]))

    zeile("Offene Positionen", str(len(konto["positionen"])))

    return konto


# ============================================================
# 16. KONTOVERMÖGEN
# ============================================================

def kontowert_berechnen(konto, kurse):

    wert = konto["guthaben"]

    for coin, position in konto["positionen"].items():

        wert += (
            position["menge"]
            * kurse[coin]
        )

    return wert


# ============================================================
# 17. RISIKO OFFENER POSITIONEN
# ============================================================

def offenes_risiko_berechnen(konto):

    gesamtrisiko = 0.0

    for position in konto["positionen"].values():

        stop_erloes = (

            position["menge"]

            * position["stop_loss"]

            * (1 - SLIPPAGE)

            * (1 - GEBUEHR)

        )

        risiko = max(
            0,
            position["gesamtkosten"] - stop_erloes
        )

        gesamtrisiko += risiko

    return gesamtrisiko


# ============================================================
# 18. RISK GATE
# ============================================================

def risk_gate(konto, coin, kurse, atr_wert):

    # Keine echten Trades

    if not PAPER_TRADING:

        return None, "Nur Paper Trading erlaubt"

    if not AUTOMATISCHE_KAEUFE:

        return None, "Automatische Käufe deaktiviert"

    if konto.get(
        "tageslimit_erreicht",
        False
    ):

        return None, "Tagesverlustlimit erreicht"

    # Keine doppelte Position

    if coin in konto["positionen"]:

        return None, "Position bereits offen"

    if (
        len(konto["positionen"])
        >= MAX_OFFENE_POSITIONEN
    ):

        return None, "Maximale Positionszahl erreicht"

    # Handelspause nach Verkauf

    letzter_ausstieg = konto[
        "letzter_ausstieg"
    ].get(coin)

    if letzter_ausstieg:

        zeitpunkt = datetime.fromisoformat(
            letzter_ausstieg
        )

        if zeitpunkt.tzinfo is None:

            zeitpunkt = zeitpunkt.replace(
                tzinfo=timezone.utc
            )

        if (
            datetime.now(timezone.utc)
            - zeitpunkt
        ).total_seconds() < HANDELSPAUSE:

            return None, "24 Stunden Handelspause"

    # ATR muss gültig sein

    if (
        not gueltige_zahl(atr_wert)
        or atr_wert <= 0
    ):

        return None, "Ungültiger ATR-Wert"

    # Gesamtes Kontovermögen

    kapital = kontowert_berechnen(
        konto,
        kurse
    )

    # Bereits gebundenes Risiko

    offenes_risiko = offenes_risiko_berechnen(
        konto
    )

    # Verfügbares Gesamtrisiko

    verfuegbares_risiko = (

        kapital * MAX_GESAMTRISIKO

        - offenes_risiko

    )

    # Maximales Risiko je Trade

    risiko_budget = min(

        kapital * MAX_RISIKO_PRO_TRADE,

        verfuegbares_risiko

    )

    if risiko_budget <= 0:

        return None, "Kein Risikobudget"

    # Simulierter Einstiegskurs

    kurs = kurse[coin]

    einstieg = kurs * (1 + SLIPPAGE)

    # ATR-basierter initialer Stop

    stop_loss = einstieg - (
        ATR_STOP_MULTIPLIKATOR * atr_wert
    )

    if stop_loss <= 0:

        return None, "Ungültiger ATR-Stop"

    # Risiko je Einheit inklusive Kosten

    stop_erloes = (

        stop_loss

        * (1 - SLIPPAGE)

        * (1 - GEBUEHR)

    )

    risiko_je_einheit = (

        einstieg * (1 + GEBUEHR)

        - stop_erloes

    )

    if risiko_je_einheit <= 0:

        return None, "Ungültige Risikoberechnung"

    # Positionsgröße nach Risiko

    menge_risiko = (

        risiko_budget

        / risiko_je_einheit

    )

    # Positionsgröße nach maximalem Positionswert

    menge_positionslimit = (

        kapital * MAX_POSITIONSANTEIL

    ) / (

        einstieg * (1 + GEBUEHR)

    )

    # Verfügbares Guthaben berücksichtigen

    menge_guthaben = (

        konto["guthaben"]

    ) / (

        einstieg * (1 + GEBUEHR)

    )

    menge = min(

        menge_risiko,

        menge_positionslimit,

        menge_guthaben

    )

    if (
        not gueltige_zahl(menge)
        or menge <= 0
    ):

        return None, "Ungültige Positionsgröße"

    geplantes_risiko = (
        menge * risiko_je_einheit
    )

    # Abschließende Risikoprüfung

    if geplantes_risiko > (
        kapital * MAX_RISIKO_PRO_TRADE + 1e-8
    ):

        return None, "Einzelrisiko überschritten"

    if (
        offenes_risiko + geplantes_risiko
        > kapital * MAX_GESAMTRISIKO + 1e-8
    ):

        return None, "Gesamtrisiko überschritten"

    return {

        "menge": menge,

        "einstieg": einstieg,

        "stop_loss": stop_loss,

        "atr_bei_einstieg": atr_wert,

        "hoechster_schlusskurs": kurs,

        "geplantes_risiko": geplantes_risiko

    }, "Freigegeben"


# ============================================================
# 19. VIRTUELLER KAUF
# ============================================================

def virtueller_kauf(konto, coin, pruefung):

    menge = pruefung["menge"]

    einstieg = pruefung["einstieg"]

    positionswert = (
        menge * einstieg
    )

    gebuehr = (
        positionswert * GEBUEHR
    )

    gesamtkosten = (
        positionswert + gebuehr
    )

    if coin in konto["positionen"]:

        print(
            faerben(
                "⚠️  Kauf abgelehnt: Position bereits offen.",
                FARBE_GELB
            )
        )

        return False

    if gesamtkosten > konto["guthaben"] + 1e-8:

        print(
            faerben(
                "⚠️  Kauf abgelehnt: Guthaben zu gering.",
                FARBE_GELB
            )
        )

        return False

    # Guthaben reduzieren

    konto["guthaben"] -= gesamtkosten

    # Position anlegen

    konto["positionen"][coin] = {

        "menge": menge,

        "einstieg": einstieg,

        "stop_loss": pruefung["stop_loss"],

        "atr_bei_einstieg": pruefung["atr_bei_einstieg"],

        "hoechster_schlusskurs": pruefung["hoechster_schlusskurs"],

        "gesamtkosten": gesamtkosten,

        "kaufzeit": zeitstempel(),

        "datenquelle": "Kraken"

    }

    # Transaktion dokumentieren

    konto["transaktionen"].append({

        "zeit": zeitstempel(),

        "coin": coin,

        "aktion": "KAUF",

        "menge": menge,

        "kurs": einstieg,

        "gebuehr": gebuehr,

        "gesamtkosten": gesamtkosten,

        "atr_bei_einstieg": pruefung["atr_bei_einstieg"],

        "datenquelle": "Kraken"

    })

    konto_speichern(konto)

    ereignis(
        "KAUF",
        coin,
        f"{euro(einstieg)} · Stop {euro(pruefung['stop_loss'])}"
    )

    abstand_stop = einstieg - pruefung["stop_loss"]

    abstand_stop_prozent = (
        abstand_stop / einstieg * 100
    )

    box_start(f"KAUF: {COINS[coin]['name']}", "✅")

    box_zeile(f"Kaufkurs        {euro(einstieg)}")

    box_zeile(f"Positionswert   {euro(positionswert)}")

    box_zeile(
        f"Risiko          "
        f"{euro(pruefung['geplantes_risiko'])}"
    )

    box_zeile(
        f"Stop-Loss       {euro(pruefung['stop_loss'])}  "
        f"(-{euro(abstand_stop)} / -{abstand_stop_prozent:.2f} %)"
    )

    box_zeile("Take-Profit     kein festes Ziel (Trailing-Stop)")

    box_zeile(
        f"ATR bei Einstieg {euro(pruefung['atr_bei_einstieg'])}"
    )

    box_ende()

    return True


# ============================================================
# 20. VIRTUELLER VERKAUF
# ============================================================

def virtueller_verkauf(
    konto,
    coin,
    referenzkurs,
    grund
):

    position = konto["positionen"][coin]

    menge = position["menge"]

    verkaufskurs = (
        referenzkurs * (1 - SLIPPAGE)
    )

    verkaufswert = (
        menge * verkaufskurs
    )

    gebuehr = (
        verkaufswert * GEBUEHR
    )

    netto_erloes = (
        verkaufswert - gebuehr
    )

    gewinn = (
        netto_erloes
        - position["gesamtkosten"]
    )

    # Guthaben aktualisieren

    konto["guthaben"] += netto_erloes

    # Position entfernen

    del konto["positionen"][coin]

    # Zeitpunkt des Verkaufs speichern

    konto["letzter_ausstieg"][coin] = (
        zeitstempel()
    )

    # Transaktion dokumentieren

    konto["transaktionen"].append({

        "zeit": zeitstempel(),

        "coin": coin,

        "aktion": "VERKAUF",

        "grund": grund,

        "menge": menge,

        "kurs": verkaufskurs,

        "gebuehr": gebuehr,

        "netto_erloes": netto_erloes,

        "gewinn": gewinn,

        "datenquelle": "Kraken"

    })

    konto_speichern(konto)

    grund_text = {
        "STOP": "Trailing-Stop ausgelöst",
        "TAGESLIMIT": "Tagesverlustlimit erreicht"
    }.get(grund, grund)

    ereignis(
        "VERKAUF",
        coin,
        f"{grund_text} · {euro(verkaufskurs)} · "
        f"Ergebnis {euro(gewinn)}"
    )

    box_start(f"VERKAUF: {COINS[coin]['name']}", "🛑")

    box_zeile(f"Grund           {grund_text}")

    box_zeile(f"Verkaufskurs    {euro(verkaufskurs)}")

    box_zeile(
        "Gewinn/Verlust  "
        + wert_faerben(gewinn, euro(gewinn))
        + f"  {pfeil(gewinn)}"
    )

    box_ende()


# ============================================================
# 21. TRAILING-STOP AKTUALISIEREN
# ============================================================

def trailing_stop_berechnen(
    position,
    letzter_schluss,
    aktueller_atr
):

    # Reine Berechnung ohne Seiteneffekt (kein Speichern) -
    # so lässt sich die Trailing-Stop-Logik im Funktionstest
    # prüfen, ohne die echte Kontodatei anzufassen.

    hoechster = position["hoechster_schlusskurs"]

    stop = position["stop_loss"]

    if (
        not gueltige_zahl(letzter_schluss)
        or not gueltige_zahl(aktueller_atr)
        or aktueller_atr <= 0
    ):

        return hoechster, stop, False

    # Trailing-Anker nur nach oben anpassen

    if letzter_schluss > hoechster:

        hoechster = letzter_schluss

    neuer_stop = (

        hoechster

        - (ATR_TRAILING_MULTIPLIKATOR * aktueller_atr)

    )

    # Stop darf nur steigen, nie sinken

    if neuer_stop > stop:

        return hoechster, neuer_stop, True

    return hoechster, stop, False


def trailing_stop_aktualisieren(
    konto,
    coin,
    letzter_schluss,
    aktueller_atr
):

    if coin not in konto["positionen"]:

        return

    position = konto["positionen"][coin]

    alter_stop = position["stop_loss"]

    alter_hoechster = position["hoechster_schlusskurs"]

    hoechster, neuer_stop, geaendert = trailing_stop_berechnen(
        position,
        letzter_schluss,
        aktueller_atr
    )

    position["hoechster_schlusskurs"] = hoechster

    if geaendert:

        position["stop_loss"] = neuer_stop

    # Auch einen neuen Höchstkurs ohne Stop-Änderung speichern
    # (V4.0 hat ihn dann verworfen, weil das Konto in jedem
    # Durchlauf neu aus der Datei geladen wird).

    if geaendert or hoechster != alter_hoechster:

        konto_speichern(konto)

    if geaendert:

        ereignis(
            "STOP_NACHGEZOGEN",
            coin,
            f"{euro(alter_stop)} → {euro(neuer_stop)}"
        )

        print(
            faerben(
                f"  ↳ 🔧 Trailing-Stop {COINS[coin]['name']} "
                f"nachgezogen: {euro(alter_stop)} → "
                f"{euro(neuer_stop)}",
                FARBE_GELB
            )
        )


# ============================================================
# 22. POSITIONEN ÜBERWACHEN
# ============================================================

def positionen_ueberwachen(konto, kurse):

    if not konto["positionen"]:

        return

    ausgestoppte_coins = []

    box_start("Positions-Check", "🔍")

    box_zeile(
        f"{'Coin':<10}{'Kurs':<12}{'Stop-Loss':<13}"
        f"{'Abstand':<10}Status"
    )

    for coin in list(
        konto["positionen"].keys()
    ):

        position = konto["positionen"][coin]

        kurs = kurse[coin]

        stop = position["stop_loss"]

        abstand_stop_prozent = (
            (kurs - stop) / kurs * 100
            if kurs else 0.0
        )

        ausgestoppt = kurs <= stop

        if ausgestoppt:

            ausgestoppte_coins.append(coin)

            status = faerben("🛑 STOP", FARBE_ROT)

        else:

            status = faerben("✅ offen", FARBE_GRUEN)

        box_zeile(
            f"{COINS[coin]['name']:<10}"
            f"{euro(kurs):<12}"
            f"{euro(stop):<13}"
            f"{abstand_stop_prozent:>5.2f} %   "
            f"{status}"
        )

    box_ende()

    # Verkäufe nach der Tabellenanzeige ausführen, damit jede
    # ausgelöste Verkaufsbox darunter erscheint statt die
    # Übersicht zu unterbrechen.

    for coin in ausgestoppte_coins:

        virtueller_verkauf(
            konto,
            coin,
            kurse[coin],
            "STOP"
        )


# ============================================================
# 23. TAGESVERLUSTLIMIT
# ============================================================

def tageslimit_pruefen(konto, kurse):

    heute = datetime.now(
        timezone.utc
    ).date().isoformat()

    kapital = kontowert_berechnen(
        konto,
        kurse
    )

    # Tagesstartwert speichern

    if konto.get("tagesdatum") != heute:

        konto["tagesdatum"] = heute

        konto["tagesstartwert"] = kapital

        konto["tageslimit_erreicht"] = False

        konto_speichern(konto)

    verlustgrenze = (

        konto["tagesstartwert"]

        * (1 - MAX_TAGESVERLUST)

    )

    if kapital <= verlustgrenze:

        if not konto["tageslimit_erreicht"]:

            konto["tageslimit_erreicht"] = True

            konto_speichern(konto)

            kasten("TAGESVERLUSTLIMIT ERREICHT", "⚠️")

            ereignis(
                "TAGESLIMIT",
                None,
                "Alle Positionen geschlossen, kein Handel bis morgen (UTC)"
            )

            print(
                faerben(
                    "Alle offenen Positionen werden "
                    "geschlossen. Kein Handel bis morgen.",
                    FARBE_ROT
                )
            )

            # Alle Positionen virtuell schließen

            for coin in list(
                konto["positionen"].keys()
            ):

                virtueller_verkauf(
                    konto,
                    coin,
                    kurse[coin],
                    "TAGESLIMIT"
                )

        return False

    return not konto["tageslimit_erreicht"]


# ============================================================
# 24. MARKTANALYSE
# ============================================================

def marktanalyse(konto, kurse):

    # Gibt je Coin ein Ergebnis zurück (nur für die Anzeige im
    # Cockpit); die Handelsentscheidungen selbst sind wie V4.0.

    kasten("AUTOMATISCHE MARKTANALYSE", "📊")

    ergebnisse = []

    for coin, info in COINS.items():

        ergebnis = {
            "coin": coin,
            "zeit": zeitstempel(),
            "status": "OK",
            "signal": None,
            "gruende": [],
            "entscheidung": ""
        }

        ergebnisse.append(ergebnis)

        # Historische Kursdaten aktualisieren

        try:

            df = historische_kurse(coin)

            marktdaten_pruefen(
                df,
                coin
            )

        except (
            requests.exceptions.RequestException,
            RuntimeError,
            ValueError
        ) as e:

            box_start(info["name"], "📉")

            box_zeile(
                faerben(
                    f"Keine neuen Trades: {e}",
                    FARBE_GELB
                )
            )

            box_ende()

            ergebnis["status"] = "DATENFEHLER"

            ergebnis["entscheidung"] = f"Keine neuen Trades: {e}"

            ereignis("DATENFEHLER", coin, str(e))

            continue

        # Indikatoren berechnen

        df["MA20"] = (
            df["close"]
            .rolling(20)
            .mean()
        )

        df["MA50"] = (
            df["close"]
            .rolling(50)
            .mean()
        )

        df["MA200"] = (
            df["close"]
            .rolling(TREND_FILTER_PERIODE)
            .mean()
        )

        df["ATR"] = atr_berechnen(
            df,
            ATR_PERIODE
        )

        df["ADX"] = adx_berechnen(
            df,
            ADX_PERIODE
        )

        aktuell = df.iloc[-1]

        ma20 = float(aktuell["MA20"])
        ma50 = float(aktuell["MA50"])
        ma200 = float(aktuell["MA200"])
        atr = float(aktuell["ATR"])
        adx = float(aktuell["ADX"])
        letzter_schluss = float(aktuell["close"])

        ergebnis.update(
            kerze_utc=df.index[-1].isoformat(),
            letzter_schluss=letzter_schluss,
            ma20=ma20,
            ma50=ma50,
            ma200=ma200,
            atr=atr,
            adx=adx
        )

        box_start(info["name"], "📈")

        if not all(
            gueltige_zahl(wert)
            for wert in [ma20, ma50, ma200, atr, adx]
        ):

            box_zeile(
                faerben(
                    "Indikatoren noch nicht vollständig "
                    "berechnet (Warmlaufphase).",
                    FARBE_GELB
                )
            )

            box_ende()

            ergebnis["status"] = "WARMLAUF"

            ergebnis["entscheidung"] = (
                "Indikatoren noch nicht vollständig berechnet"
            )

            continue

        kurs = kurse[coin]

        ergebnis["kurs"] = kurs

        box_zeile(f"Kurs            {euro(kurs)}")

        box_zeile(
            f"MA20/50/200     {euro(ma20)} / "
            f"{euro(ma50)} / {euro(ma200)}"
        )

        box_zeile(f"ATR14 / ADX14   {euro(atr)} / {adx:.1f}")

        # Trailing-Stop einer bestehenden Position
        # unabhängig vom Signal aktualisieren

        trailing_stop_aktualisieren(
            konto,
            coin,
            letzter_schluss,
            atr
        )

        # Signalprüfung

        trend_intakt = ma50 > ma200

        crossover_long = (
            ma20 > ma50
            and kurs > ma20
        )

        trend_stark = adx >= ADX_MINDESTWERT

        if (
            trend_intakt
            and crossover_long
            and trend_stark
        ):

            signal = "LONG"

        else:

            signal = "NO TRADE"

        box_zeile(f"Signal          {signal_text(signal)}")

        ergebnis["signal"] = signal

        if signal != "LONG":

            gruende = []

            if not trend_intakt:

                gruende.append("Trendfilter (MA50<=MA200)")

            if not crossover_long:

                gruende.append("kein Crossover")

            if not trend_stark:

                gruende.append(f"ADX < {ADX_MINDESTWERT}")

            box_zeile(f"  ↳ {', '.join(gruende)}")

            box_ende()

            ergebnis["gruende"] = gruende

            ergebnis["entscheidung"] = "Kein Einstieg"

            continue

        box_ende()

        # Risikoprüfung

        pruefung, grund = risk_gate(
            konto,
            coin,
            kurse,
            atr
        )

        if pruefung is None:

            print(
                faerben(
                    f"  ⚠️  Kein Kauf ({info['name']}): {grund}",
                    FARBE_GELB
                )
            )

            ergebnis["entscheidung"] = f"Kein Kauf: {grund}"

            # "Position bereits offen" ist bei einem laufenden
            # Trend der Normalfall und kein Ereignis.

            if grund != "Position bereits offen":

                ereignis("KEIN_KAUF", coin, grund)

            continue

        # Virtuellen Kauf ausführen

        gekauft = virtueller_kauf(
            konto,
            coin,
            pruefung
        )

        ergebnis["entscheidung"] = (
            "Virtuell gekauft" if gekauft
            else "Kauf abgelehnt"
        )

    analyse_speichern(ergebnisse)

    return ergebnisse


def analyse_speichern(ergebnisse):

    # Nur Anzeige - ein Schreibfehler darf den Handel nicht
    # beeinflussen.

    if ANALYSE_DATEI is None:

        return

    try:

        temporaer = Path(
            str(ANALYSE_DATEI) + ".tmp"
        )

        temporaer.write_text(
            json.dumps(ergebnisse, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        os.replace(
            temporaer,
            ANALYSE_DATEI
        )

    except OSError as e:

        print(
            faerben(
                f"Marktanalyse nicht gespeichert: {type(e).__name__}",
                FARBE_GRAU
            )
        )


def analyse_laden():

    if ANALYSE_DATEI is None or not ANALYSE_DATEI.exists():

        return []

    try:

        daten = json.loads(
            ANALYSE_DATEI.read_text(encoding="utf-8")
        )

    except (OSError, ValueError):

        return []

    return daten if isinstance(daten, list) else []


# ============================================================
# 25. STATUSANZEIGE
# ============================================================

def status_anzeigen(konto, kurse):

    kapital = kontowert_berechnen(
        konto,
        kurse
    )

    gewinn = (
        kapital - konto["startkapital"]
    )

    rendite = (
        gewinn
        / konto["startkapital"]
        * 100
    )

    offenes_risiko = offenes_risiko_berechnen(konto)

    risiko_anteil = (
        offenes_risiko / kapital
        if kapital else 0.0
    )

    # --- Portfolio-Übersicht ---

    box_start("Portfolio", "💰")

    box_zeile(f"Gesamtvermögen     {euro(kapital)}")

    box_zeile(
        "Gewinn/Verlust     "
        + wert_faerben(gewinn, f"{euro(gewinn)}")
        + f"  ({rendite:+.2f} %)  {pfeil(gewinn)}"
    )

    box_zeile(f"Guthaben (frei)    {euro(konto['guthaben'])}")

    box_zeile(
        f"Offene Positionen  "
        f"{len(konto['positionen'])} / {MAX_OFFENE_POSITIONEN}"
    )

    box_zeile(
        f"Risiko genutzt     "
        f"{balken(risiko_anteil / MAX_GESAMTRISIKO)} "
        f"{risiko_anteil * 100:.2f} % / "
        f"{MAX_GESAMTRISIKO * 100:.0f} %"
    )

    box_ende()

    # --- Positionstabelle ---

    if konto["positionen"]:

        box_start("Offene Positionen", "📋")

        box_zeile(
            f"{'Coin':<10}{'Kurs':<11}{'Einstieg':<11}"
            f"{'P&L':<9}{'Stop-Loss':<11}Abstand"
        )

        for coin, position in konto["positionen"].items():

            kurs = kurse[coin]

            stop = position["stop_loss"]

            abstand_stop_prozent = (
                (kurs - stop) / kurs * 100
                if kurs else 0.0
            )

            unrealisiert_prozent = (
                (kurs - position["einstieg"])
                / position["einstieg"] * 100
            )

            box_zeile(
                spalte(COINS[coin]["name"], 10)
                + spalte(euro(kurs), 11)
                + spalte(euro(position["einstieg"]), 11)
                + spalte(
                    wert_faerben(
                        unrealisiert_prozent,
                        f"{unrealisiert_prozent:+.2f} %"
                    ),
                    9
                )
                + spalte(euro(stop), 11)
                + f"{abstand_stop_prozent:.2f} %"
            )

        box_ende()

        print(
            faerben(
                "ℹ️  Take-Profit: kein festes Ziel — "
                "Exit ausschließlich über Trailing-Stop",
                FARBE_GRAU
            )
        )


# ============================================================
# 26. AUTOMATISCHER FUNKTIONSTEST
# ============================================================

def smoke_test():

    unterkasten("Funktionstest", "🧪")

    # --- Test 1: Risk Gate / Positionsberechnung ---

    testkonto = neues_konto()

    testkurse = {

        "bitcoin": 70000.0,

        "ethereum": 2500.0,

        "solana": 120.0

    }

    test_atr = 1200.0

    pruefung, grund = risk_gate(
        testkonto,
        "bitcoin",
        testkurse,
        test_atr
    )

    if pruefung is None:

        raise RuntimeError(
            f"Risk-Gate-Test fehlgeschlagen: {grund}"
        )

    assert pruefung["menge"] > 0

    assert pruefung["stop_loss"] < pruefung["einstieg"]

    assert pruefung["geplantes_risiko"] <= (

        STARTKAPITAL
        * MAX_RISIKO_PRO_TRADE
        + 1e-8

    )

    print(faerben("  ✅ Risk Gate", FARBE_GRUEN))

    print(faerben("  ✅ Positionsberechnung", FARBE_GRUEN))

    # --- Test 2: ATR / ADX auf synthetischen Daten ---

    anzahl = 300

    index = pd.date_range(
        "2025-01-01",
        periods=anzahl,
        freq="h",
        tz="UTC"
    )

    # Als NumPy-Array erzeugen, nicht als pd.Series mit eigenem
    # RangeIndex - sonst richtet Pandas beim DataFrame-Bau nach
    # Index aus (RangeIndex vs. DatetimeIndex) und alles wird NaN.

    basis = (
        pd.RangeIndex(anzahl).to_numpy(dtype=float)
        * 5
        + 50000
    )

    test_df = pd.DataFrame({

        "open": basis,

        "high": basis + 80,

        "low": basis - 80,

        "close": basis + 20,

        "volume": 1.0

    }, index=index)

    test_atr_serie = atr_berechnen(
        test_df,
        ATR_PERIODE
    )

    test_adx_serie = adx_berechnen(
        test_df,
        ADX_PERIODE
    )

    letzter_atr = float(
        test_atr_serie.iloc[-1]
    )

    letzter_adx = float(
        test_adx_serie.iloc[-1]
    )

    assert gueltige_zahl(letzter_atr) and letzter_atr > 0

    assert gueltige_zahl(letzter_adx) and 0 <= letzter_adx <= 100

    print(faerben("  ✅ ATR-Berechnung", FARBE_GRUEN))

    print(faerben("  ✅ ADX-Berechnung", FARBE_GRUEN))

    # --- Test 3: Trailing-Stop zieht nur nach oben nach ---
    # Reine Funktion ohne Speichern - die echte Kontodatei
    # bleibt beim Funktionstest unberührt.

    testposition = {

        "menge": 0.01,

        "einstieg": 70000.0,

        "stop_loss": 67600.0,

        "atr_bei_einstieg": 1200.0,

        "hoechster_schlusskurs": 70000.0,

        "gesamtkosten": 700.0

    }

    hoechster, neuer_stop, geaendert = trailing_stop_berechnen(
        testposition,
        letzter_schluss=72000.0,
        aktueller_atr=1000.0
    )

    assert geaendert is True

    assert neuer_stop > 67600.0

    testposition["hoechster_schlusskurs"] = hoechster

    testposition["stop_loss"] = neuer_stop

    # Fallender Kurs darf den Stop nicht zurückziehen

    hoechster2, neuer_stop2, geaendert2 = trailing_stop_berechnen(
        testposition,
        letzter_schluss=69000.0,
        aktueller_atr=1000.0
    )

    assert geaendert2 is False

    assert neuer_stop2 == neuer_stop

    print(faerben("  ✅ Trailing-Stop-Logik", FARBE_GRUEN))

    # --- Test 4: Stop über dem Einstieg ist ein gültiger ---
    # Kontostand (V4.0 hat hier beim Speichern abgebrochen)

    testkonto = neues_konto()

    testposition = dict(testposition)

    testposition["stop_loss"] = testposition["einstieg"] + 500.0

    testkonto["positionen"]["bitcoin"] = testposition

    konto_pruefen(testkonto)

    assert offenes_risiko_berechnen(testkonto) == 0

    print(faerben("  ✅ Trailing-Stop über Einstieg", FARBE_GRUEN))


# ============================================================
# 27. COCKPIT-ANBINDUNG
# ============================================================

def neuer_lauf():

    # Zustand des laufenden Bots nur für die Anzeige. Die
    # Handelsentscheidungen hängen davon nicht ab.

    return {
        "konto": None,
        "kurse": None,
        "kurse_zeit_utc": None,
        "analyse": analyse_laden(),
        "gestartet_utc": zeitstempel(),
        "letzte_pruefung_utc": None,
        "naechste_pruefung_utc": None,
        "durchlaeufe": 0,
        "api_fehler": 0,
        "letzter_fehler": None,
        "cockpit_fehler": None
    }


def cockpit_aktualisieren(cockpit, lauf, stufe):

    # Das Cockpit liest nur, was dieser Durchlauf bereits
    # berechnet hat. Ein Fehler in der Anzeige darf den
    # Handel nie stoppen oder verändern: dann wird das
    # Cockpit für den Rest des Laufs abgeschaltet.

    if cockpit is None or lauf["cockpit_fehler"] is not None:

        return

    try:

        cockpit.update(
            lauf,
            stufe,
            list(EREIGNISSE)
        )

    except Exception as e:

        lauf["cockpit_fehler"] = type(e).__name__

        ereignis("COCKPIT_FEHLER", None, type(e).__name__)

        print(
            faerben(
                f"Cockpit konnte nicht aktualisiert werden "
                f"({type(e).__name__}) - der Handel läuft "
                "unverändert weiter.",
                FARBE_GELB
            )
        )


# ============================================================
# 28. EIN DURCHLAUF (alle 5 Minuten)
# ============================================================

def ein_durchlauf(lauf):

    # Intern wird mit UTC gerechnet (Tageslimit-Reset,
    # 24h-Handelspause, Kerzengrenzen) - für die Anzeige
    # wird nur die Textausgabe nach Europe/Berlin
    # (MESZ/MEZ) umgerechnet.

    jetzt = datetime.now(
        timezone.utc
    )

    lauf["letzte_pruefung_utc"] = jetzt.isoformat()

    kasten(
        f"KURSPRÜFUNG · {lokale_zeit_anzeigen(jetzt)}",
        "🕒"
    )

    # ----------------------------------------
    # 1. Konto laden
    # ----------------------------------------

    konto = konto_laden()

    lauf["konto"] = konto

    # ----------------------------------------
    # 2. Aktuelle Kraken-Kurse abrufen
    # ----------------------------------------

    kurse = aktuelle_kurse()

    lauf["kurse"] = kurse

    lauf["kurse_zeit_utc"] = jetzt.isoformat()

    # ----------------------------------------
    # 3. Tagesstartwert festhalten
    # ----------------------------------------

    heute = jetzt.date().isoformat()

    if konto.get("tagesdatum") != heute:

        konto["tagesdatum"] = heute

        konto["tagesstartwert"] = (
            kontowert_berechnen(
                konto,
                kurse
            )
        )

        konto["tageslimit_erreicht"] = False

        konto_speichern(konto)

    # ----------------------------------------
    # 4. Bestehende Positionen überwachen
    # ----------------------------------------

    positionen_ueberwachen(
        konto,
        kurse
    )

    # ----------------------------------------
    # 5. Tagesverlustlimit prüfen
    # ----------------------------------------

    handel_erlaubt = tageslimit_pruefen(
        konto,
        kurse
    )

    # ----------------------------------------
    # 6. Stündliche Marktanalyse
    # ----------------------------------------

    aktuelle_stunde = (
        jetzt.strftime("%Y-%m-%d-%H")
    )

    if (
        handel_erlaubt
        and konto.get("letzte_analyse")
        != aktuelle_stunde
    ):

        # Analysezeitpunkt speichern

        konto["letzte_analyse"] = (
            aktuelle_stunde
        )

        konto_speichern(konto)

        # Marktanalyse durchführen

        lauf["analyse"] = marktanalyse(
            konto,
            kurse
        )

    # ----------------------------------------
    # 7. Kontostand anzeigen
    # ----------------------------------------

    status_anzeigen(
        konto,
        kurse
    )

    print(
        faerben(
            "\nNächste Prüfung in ca. 5 Minuten.",
            FARBE_GRAU
        )
    )


# ============================================================
# 29. HAUPTSCHLEIFE
# ============================================================

def hauptschleife(cockpit=None, durchlaeufe=None, warten=time.sleep):

    # durchlaeufe/warten nur für Tests; im Betrieb läuft die
    # Schleife, bis sie gestoppt wird.

    lauf = neuer_lauf()

    try:

        while True:

            try:

                ein_durchlauf(lauf)

                stufe = "WARTET"

            except requests.exceptions.RequestException as e:

                print(
                    faerben(f"\n⚠️  API-FEHLER: {e}", FARBE_GELB)
                )

                print(
                    faerben(
                        "Keine Handelsausführung in diesem Durchlauf.",
                        FARBE_GELB
                    )
                )

                lauf["api_fehler"] += 1

                lauf["letzter_fehler"] = str(e)

                ereignis("API_FEHLER", None, str(e))

                stufe = "API_FEHLER"

            except Exception as e:

                kasten("KRITISCHER PROGRAMMFEHLER", "🔥")

                traceback.print_exc()

                print(
                    faerben(
                        "\nDer Bot wird aus Sicherheitsgründen "
                        "gestoppt.",
                        FARBE_ROT
                    )
                )

                lauf["letzter_fehler"] = (
                    f"{type(e).__name__}: {e}"
                )

                ereignis(
                    "PROGRAMMFEHLER",
                    None,
                    lauf["letzter_fehler"]
                )

                cockpit_aktualisieren(cockpit, lauf, "FEHLER")

                raise

            lauf["durchlaeufe"] += 1

            lauf["naechste_pruefung_utc"] = (
                datetime.now(timezone.utc)
                + timedelta(seconds=INTERVALL)
            ).isoformat()

            cockpit_aktualisieren(cockpit, lauf, stufe)

            if (
                durchlaeufe is not None
                and lauf["durchlaeufe"] >= durchlaeufe
            ):

                return lauf

            # Fünf Minuten warten

            warten(INTERVALL)

    except KeyboardInterrupt:

        kasten(f"TRADING BOT V{VERSION} GESTOPPT", "🛑")

        lauf["naechste_pruefung_utc"] = None

        cockpit_aktualisieren(cockpit, lauf, "GESTOPPT")

        print(
            faerben(
                "Das virtuelle Handelskonto bleibt in "
                "Google Drive gespeichert.",
                FARBE_GRAU
            )
        )

    return lauf


# ============================================================
# 30. START
# ============================================================

def start(
    cockpit=None,
    ordner=STANDARD_ORDNER,
    drive_verbinden=True
):

    global HANDELSPAARE

    kasten(f"TRADING BOT V{VERSION}", "🤖")

    print(
        faerben(
            "Kraken Paper Trading · Trendfolge-Strategie".center(BREITE),
            FARBE_GRAU
        )
    )

    if drive_verbinden:

        from google.colab import drive

        drive.mount("/content/drive")

    speicherorte_einrichten(ordner)

    HANDELSPAARE = handelspaare_ermitteln()

    konto_uebernehmen()

    if not PAPER_TRADING:

        raise RuntimeError(
            "Dieses Programm darf nur virtuell handeln."
        )

    # Zuerst alle Kernfunktionen überprüfen

    smoke_test()

    kasten(f"TRADING BOT V{VERSION} GESTARTET", "🚀")

    zeile("Handelsmodus", "PAPER TRADING", "📝")
    zeile("Datenquelle", "Kraken", "🔗")
    zeile("Kontowährung", "EUR", "💶")
    zeile(
        "Strategie",
        "Trendfolge (MA20/50/200, ADX, ATR-Trailing-Stop)",
        "📈"
    )
    zeile("Marktanalyse", "stündlich", "🕐")
    zeile("Positionsüberwachung", "alle 5 Minuten", "🔍")
    zeile(
        "Automatische Käufe",
        "aktiv" if AUTOMATISCHE_KAEUFE else "deaktiviert",
        "🤖"
    )
    zeile(
        "Cockpit",
        "aktiv" if cockpit is not None else "aus",
        "🖥️"
    )

    trenner()

    return hauptschleife(cockpit)


In [ ]:
%%writefile /content/crypto_cockpit_v41.py
"""Read-only cockpit for the Kraken paper-trading bot V4.1 (Colab view, browser tab, Drive file).

Shows only what the bot has already computed in the current pass (account, prices of the
last price check, last hourly market analysis, events). It never fetches market data and
never changes the account."""
from datetime import datetime, timedelta, timezone
from html import escape
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
import math
import threading

import crypto_bot_v41 as bot


def safe(value):
    return escape(str(value if value is not None else '—'), quote=True)


def finite(value):
    return (isinstance(value, (int, float)) and not isinstance(value, bool)
            and math.isfinite(value))


def euro(value):
    return bot.euro(value) if finite(value) else '—'


def signed_euro(value):
    if not finite(value):
        return '—'
    return ('+' if value > 0 else '') + bot.euro(value)


def number(value, digits=2):
    if not finite(value):
        return '—'
    text = f'{value:,.{digits}f}'
    return text.replace(',', 'X').replace('.', ',').replace('X', '.')


def percent(value, digits=2, sign=True):
    if not finite(value):
        return '—'
    return (f'{value:+.{digits}f}' if sign else f'{value:.{digits}f}').replace('.', ',') + ' %'


def parse_utc(value):
    if value is None:
        return None
    moment = datetime.fromisoformat(value) if isinstance(value, str) else value
    if moment.tzinfo is None:
        moment = moment.replace(tzinfo=timezone.utc)
    return moment.astimezone(timezone.utc)


def berlin(value, with_date=True):
    """Short local time for tables; the text log uses bot.lokale_zeit_anzeigen()."""
    moment = parse_utc(value)
    if moment is None:
        return '—'
    local = moment.astimezone(bot.BERLIN_ZEITZONE)
    return local.strftime('%d.%m. %H:%M' if with_date else '%H:%M')


def tone(value):
    if not finite(value) or value == 0:
        return 'neutral'
    return 'ok' if value > 0 else 'bad'


def table(headings, rows, empty):
    header = ''.join(f'<th>{safe(h)}</th>' for h in headings)
    body = ''.join('<tr>' + ''.join(f'<td>{cell}</td>' for cell in row) + '</tr>'
                   for row in rows)
    if not body:
        body = f'<tr><td colspan="{len(headings)}" class="muted">{safe(empty)}</td></tr>'
    return f'<div class="scroll"><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'


# ---------- snapshot: numbers from the bot's own state, no market data ----------

def liquidation_value(menge, kurs):
    """Net proceeds of a virtual sale at `kurs`, with the bot's simulated costs."""
    return menge * kurs * (1 - bot.SLIPPAGE) * (1 - bot.GEBUEHR)


def closed_trades(transaktionen, limit=12):
    """Pair each VERKAUF with the preceding KAUF of the same coin."""
    offene_kaeufe = {}
    trades = []
    for t in transaktionen:
        if t.get('aktion') == 'KAUF':
            offene_kaeufe[t.get('coin')] = t
        elif t.get('aktion') == 'VERKAUF':
            kauf = offene_kaeufe.pop(t.get('coin'), {})
            trades.append(dict(coin=t.get('coin'), menge=t.get('menge'),
                               kaufzeit=kauf.get('zeit'), verkaufszeit=t.get('zeit'),
                               kaufkurs=kauf.get('kurs'), verkaufskurs=t.get('kurs'),
                               gewinn=t.get('gewinn'), grund=t.get('grund')))
    return trades[-limit:], len(trades)


def trading_pause_until(konto, coin, now):
    letzter = konto.get('letzter_ausstieg', {}).get(coin)
    if not letzter:
        return None
    ende = parse_utc(letzter) + timedelta(seconds=bot.HANDELSPAUSE)
    return ende.isoformat() if ende > now else None


def snapshot(lauf, stufe, ereignisse, now=None):
    now = parse_utc(now) or datetime.now(timezone.utc)
    konto, kurse = lauf.get('konto'), lauf.get('kurse')
    data = dict(version=bot.VERSION, stufe=stufe, paper_trading=bot.PAPER_TRADING,
                echte_orders=0, aktualisiert_utc=now.isoformat(),
                gestartet_utc=lauf.get('gestartet_utc'),
                letzte_pruefung_utc=lauf.get('letzte_pruefung_utc'),
                naechste_pruefung_utc=lauf.get('naechste_pruefung_utc'),
                kurse_zeit_utc=lauf.get('kurse_zeit_utc'),
                durchlaeufe=lauf.get('durchlaeufe', 0),
                api_fehler=lauf.get('api_fehler', 0),
                letzter_fehler=lauf.get('letzter_fehler'),
                intervall_sekunden=bot.INTERVALL,
                max_positionen=bot.MAX_OFFENE_POSITIONEN,
                max_gesamtrisiko=bot.MAX_GESAMTRISIKO,
                max_tagesverlust=bot.MAX_TAGESVERLUST,
                bewertet=False, positionen=[], abschluesse=[], abschluesse_gesamt=0,
                analyse=[], ereignisse=[dict(e) for e in list(ereignisse)[-10:]])
    if konto is None or kurse is None or any(c not in kurse for c in konto['positionen']):
        # Before the first successful price check there is nothing to value.
        return data

    kapital = bot.kontowert_berechnen(konto, kurse)
    offenes_risiko = bot.offenes_risiko_berechnen(konto)
    positionen = []
    for coin, p in konto['positionen'].items():
        kurs = kurse[coin]
        netto = liquidation_value(p['menge'], kurs)
        stop_erloes = liquidation_value(p['menge'], p['stop_loss'])
        positionen.append(dict(
            coin=coin, name=bot.COINS[coin]['name'], menge=p['menge'],
            einstieg=p['einstieg'], kaufzeit=p.get('kaufzeit'), kurs=kurs,
            marktwert=p['menge'] * kurs, gesamtkosten=p['gesamtkosten'],
            ergebnis_nach_kosten=netto - p['gesamtkosten'],
            kursaenderung_prozent=(kurs / p['einstieg'] - 1) * 100,
            stop=p['stop_loss'], stop_zu_einstieg=p['stop_loss'] - p['einstieg'],
            stop_zu_einstieg_prozent=(p['stop_loss'] / p['einstieg'] - 1) * 100,
            abstand_kurs_stop_prozent=(kurs - p['stop_loss']) / kurs * 100,
            ergebnis_am_stop=stop_erloes - p['gesamtkosten'],
            hoechster_schlusskurs=p['hoechster_schlusskurs'],
            atr_bei_einstieg=p['atr_bei_einstieg'],
            stop_erreicht=kurs <= p['stop_loss']))

    verkaeufe = [t for t in konto['transaktionen'] if t.get('aktion') == 'VERKAUF']
    abschluesse, anzahl = closed_trades(konto['transaktionen'])
    tagesstart = konto.get('tagesstartwert')
    analyse = []
    for eintrag in lauf.get('analyse') or []:
        eintrag = dict(eintrag)
        coin = eintrag.get('coin')
        eintrag['name'] = bot.COINS.get(coin, {}).get('name', coin)
        eintrag['handelspause_bis'] = trading_pause_until(konto, coin, now)
        analyse.append(eintrag)

    data.update(
        bewertet=True,
        kapital=kapital, startkapital=konto['startkapital'],
        gewinn_gesamt=kapital - konto['startkapital'],
        rendite_prozent=(kapital / konto['startkapital'] - 1) * 100,
        guthaben=konto['guthaben'],
        marktwert_positionen=sum(p['marktwert'] for p in positionen),
        offenes_ergebnis=sum(p['ergebnis_nach_kosten'] for p in positionen),
        realisiert=sum(t.get('gewinn', 0) for t in verkaeufe),
        gebuehren=sum(t.get('gebuehr', 0) for t in konto['transaktionen']),
        offenes_risiko=offenes_risiko,
        risiko_anteil=offenes_risiko / kapital if kapital else 0.0,
        tagesdatum=konto.get('tagesdatum'), tagesstartwert=tagesstart,
        tagesergebnis=kapital - tagesstart if finite(tagesstart) else None,
        tagesergebnis_prozent=(kapital / tagesstart - 1) * 100 if finite(tagesstart) and tagesstart else None,
        tagesgrenze=tagesstart * (1 - bot.MAX_TAGESVERLUST) if finite(tagesstart) else None,
        tageslimit_erreicht=bool(konto.get('tageslimit_erreicht')),
        positionen=positionen, abschluesse=abschluesse, abschluesse_gesamt=anzahl,
        analyse=analyse)
    return data


# ---------- HTML ----------

STATUS_TEXT = {
    'WARTET': ('Bot läuft · wartet auf die nächste Kursprüfung', 'ok'),
    'API_FEHLER': ('Kraken nicht erreichbar · kein Handel in diesem Durchlauf, nächster Versuch folgt', 'bad'),
    'GESTOPPT': ('Bot gestoppt', 'neutral'),
    'FEHLER': ('Programmfehler · Bot aus Sicherheitsgründen gestoppt', 'bad'),
}
# No price check is expected in these stages: an old timestamp is not a stall.
QUIET_STAGES = ('GESTOPPT', 'FEHLER')


def signal_cell(eintrag):
    status = eintrag.get('status')
    if status == 'DATENFEHLER':
        return '<span class="bad">Datenfehler</span>'
    if status == 'WARMLAUF':
        return '<span class="neutral">Warmlauf</span>'
    if eintrag.get('signal') == 'LONG':
        return '<span class="ok">LONG</span>'
    return '<span class="muted">kein Trade</span>'


def render_html(data):
    """One self-contained HTML surface; every text from outside is escaped."""
    status, status_class = STATUS_TEXT.get(data['stufe'], (data['stufe'], 'neutral'))
    if data['bewertet'] and data['tageslimit_erreicht']:
        entry_status = ('bad', 'Neue Käufe gesperrt: Tagesverlustlimit erreicht (bis Tageswechsel UTC)')
    elif data['stufe'] in QUIET_STAGES:
        entry_status = ('neutral', 'Keine neuen Käufe: Bot läuft nicht')
    else:
        entry_status = ('neutral', 'Neue Käufe: nach Signal und Risk Gate bei der stündlichen Marktanalyse')

    def tile(label, value, sub=''):
        sub = f'<small>{sub}</small>' if sub else ''
        return f'<div class="tile"><span>{safe(label)}</span><strong>{value}</strong>{sub}</div>'

    if data['bewertet']:
        risk_share = data['risiko_anteil'] / data['max_gesamtrisiko'] if data['max_gesamtrisiko'] else 0
        bar = (f'<div class="bar"><i style="width:{max(0, min(100, risk_share * 100)):.0f}%"'
               f' class="{"bad" if risk_share >= 0.8 else ""}"></i></div>')
        day_sub = (f'Grenze {euro(data["tagesgrenze"])} (−{percent(data["max_tagesverlust"] * 100, 0, False)})'
                   if data['tagesgrenze'] is not None else '')
        cards = ''.join((
            tile('Gesamtvermögen · Modell', euro(data['kapital']),
                 f'Start {euro(data["startkapital"])}'),
            tile('Gewinn/Verlust seit Start',
                 f'<span class="{tone(data["gewinn_gesamt"])}">{signed_euro(data["gewinn_gesamt"])}</span>',
                 percent(data['rendite_prozent'])),
            tile('Heute (UTC-Tag)',
                 f'<span class="{tone(data["tagesergebnis"])}">{signed_euro(data["tagesergebnis"])}</span>',
                 day_sub),
            tile('Freies Guthaben', euro(data['guthaben'])),
            tile('In Positionen · Marktwert', euro(data['marktwert_positionen']),
                 f'offen nach Kosten {signed_euro(data["offenes_ergebnis"])}'),
            tile('Offenes Risiko bis Stop', euro(data['offenes_risiko']),
                 f'{percent(data["risiko_anteil"] * 100, 2, False)} von '
                 f'{percent(data["max_gesamtrisiko"] * 100, 0, False)}{bar}'),
            tile('Realisiert', f'<span class="{tone(data["realisiert"])}">{signed_euro(data["realisiert"])}</span>',
                 f'Gebühren gesamt {euro(data["gebuehren"])}'),
            tile('Positionen offen / max · Trades',
                 f'{len(data["positionen"])} / {data["max_positionen"]}',
                 f'{data["abschluesse_gesamt"]} abgeschlossen'),
        ))
    else:
        cards = tile('Konto', 'noch nicht bewertet', 'Warte auf die erste erfolgreiche Kursprüfung')

    pos_rows = []
    for p in data['positionen']:
        stop_tone = tone(p['stop_zu_einstieg'])
        at_stop = p['ergebnis_am_stop']
        stop_note = (f'sichert {signed_euro(at_stop)} nach Kosten' if at_stop > 0
                     else f'Risiko bis Stop {euro(-at_stop)}')
        state = ('<span class="bad">Stop erreicht</span>' if p['stop_erreicht']
                 else f'{percent(p["abstand_kurs_stop_prozent"], 2, False)} über Stop')
        pos_rows.append((
            f'<b>{safe(p["name"])}</b><br><small>{number(p["menge"], 6)}</small>',
            f'{euro(p["einstieg"])}<br><small>{berlin(p["kaufzeit"])}</small>',
            f'{euro(p["kurs"])}<br><small>{state}</small>',
            f'<span class="{tone(p["ergebnis_nach_kosten"])}">{signed_euro(p["ergebnis_nach_kosten"])}</span>'
            f'<br><small>Kurs {percent(p["kursaenderung_prozent"])} ggü. Einstieg</small>',
            f'<span class="{stop_tone}">{euro(p["stop"])} ({percent(p["stop_zu_einstieg_prozent"])})</span>'
            f'<br><small>{stop_note}</small>',
            f'{euro(p["hoechster_schlusskurs"])}<br><small>ATR Einstieg {euro(p["atr_bei_einstieg"])}</small>'))

    grund_text = {'STOP': 'Trailing-Stop', 'TAGESLIMIT': 'Tagesverlustlimit'}
    trade_rows = [(
        safe(bot.COINS.get(t['coin'], {}).get('name', t['coin'])),
        f'{berlin(t["kaufzeit"])} → {berlin(t["verkaufszeit"])}',
        number(t['menge'], 6), euro(t['kaufkurs']), euro(t['verkaufskurs']),
        f'<span class="{tone(t["gewinn"])}">{signed_euro(t["gewinn"])}</span>',
        safe(grund_text.get(t['grund'], t['grund'])))
        for t in reversed(data['abschluesse'])]

    analysis_rows = []
    for a in data['analyse']:
        decision = safe(a.get('entscheidung') or '')
        if a.get('gruende'):
            decision += f'<br><small>{safe(", ".join(a["gruende"]))}</small>'
        pause = a.get('handelspause_bis')
        if pause:
            decision += f'<br><small>Handelspause bis {berlin(pause)}</small>'
        adx = a.get('adx')
        adx_text = number(adx, 1)
        if finite(adx):
            adx_text = f'<span class="{"ok" if adx >= bot.ADX_MINDESTWERT else "muted"}">{adx_text}</span>'
        analysis_rows.append((
            f'<b>{safe(a.get("name"))}</b><br><small>Kerze {berlin(a.get("kerze_utc"))}</small>',
            euro(a.get('kurs')),
            f'{euro(a.get("ma20"))}<br>{euro(a.get("ma50"))}<br>{euro(a.get("ma200"))}',
            f'{adx_text}<br><small>ATR {euro(a.get("atr"))}</small>',
            signal_cell(a), decision))

    event_rows = [(berlin(e.get('zeit')),
                   safe(bot.COINS.get(e.get('coin'), {}).get('name', e.get('coin'))),
                   safe(e.get('art')), safe(e.get('text')))
                  for e in reversed(data['ereignisse'])]

    analysis_time = next((a.get('zeit') for a in data['analyse'] if a.get('zeit')), None)
    error_note = ''
    if data['letzter_fehler'] and data['stufe'] in ('API_FEHLER', 'FEHLER'):
        error_note = f'<br><span class="bad">Letzter Fehler: {safe(data["letzter_fehler"])}</span>'
    updated_ms = int(parse_utc(data['aktualisiert_utc']).timestamp() * 1000)
    stall_minutes = 2 * data['intervall_sekunden'] // 60 + 1
    return f'''<!doctype html><html lang="de"><head><meta charset="utf-8"><style>
.kc{{font:14px/1.42 system-ui,-apple-system,sans-serif;color:#182338;background:#f5f8fc;
padding:18px;border-radius:14px;max-width:1260px;margin:0 auto}}
.kc h2{{margin:0 0 6px;font-size:22px}} .kc h3{{font-size:16px;margin:18px 0 7px}}
.kc .subtitle,.kc small,.kc .muted{{color:#58677a}}
.kc .ok{{color:#177245;font-weight:700}} .kc .bad{{color:#ba3042;font-weight:700}}
.kc .neutral{{color:#344e75;font-weight:700}}
.kc .tiles{{display:grid;grid-template-columns:repeat(auto-fit,minmax(155px,1fr));gap:9px;margin:14px 0}}
.kc .tile{{background:white;border:1px solid #d9e1ed;border-radius:10px;padding:10px 12px}}
.kc .tile>span{{display:block;color:#58677a;font-size:12px}} .kc .tile strong{{font-size:17px;display:block}}
.kc .bar{{height:6px;background:#e1e7f0;border-radius:3px;margin-top:5px;overflow:hidden}}
.kc .bar i{{display:block;height:100%;background:#177245}} .kc .bar i.bad{{background:#ba3042}}
.kc .scroll{{overflow-x:auto}} .kc table{{border-collapse:collapse;width:100%;background:white}}
.kc th,.kc td{{border-bottom:1px solid #e1e7f0;text-align:left;padding:8px 9px;vertical-align:top;white-space:nowrap}}
.kc th{{background:#eaf0fa;font-size:12px}} .kc .note{{margin-top:13px;font-size:12px;color:#47566d}}
</style></head><body><div class="kc">
<h2>Krypto-Bot V{safe(data['version'])} · Paper-Trading-Cockpit</h2>
<div class="subtitle" id="kc-stand" data-updated-ms="{updated_ms}" data-quiet="{'1' if data['stufe'] in QUIET_STAGES else '0'}" data-stall-min="{stall_minutes}">Stand {berlin(data['aktualisiert_utc'])} ·
Kurse vom {berlin(data['kurse_zeit_utc'])} · nächste Prüfung ca. {berlin(data['naechste_pruefung_utc'], False)} ·
letzte Marktanalyse {berlin(analysis_time)} · Zeiten Europe/Berlin</div>
<p><span class="{status_class}">{safe(status)}</span> · Kraken · BTC/ETH/SOL in EUR ·
PAPER TRADING · echte Orders: 0 · {data['durchlaeufe']} Prüfungen, {data['api_fehler']} API-Fehler seit Start{error_note}<br>
<span class="{entry_status[0]}">{safe(entry_status[1])}</span></p>
<div class="tiles">{cards}</div>
<h3>Offene virtuelle Positionen</h3>
{table(('Coin / Menge', 'Einstieg', 'Kurs · Abstand', 'Ergebnis nach Kosten', 'Trailing-Stop', 'Höchster Schluss'), pos_rows, 'Keine offene Position.')}
<h3>Letzte Marktanalyse (stündlich)</h3>
{table(('Coin', 'Kurs', 'MA20 / MA50 / MA200', f'ADX (≥ {bot.ADX_MINDESTWERT}) · ATR', 'Signal', 'Entscheidung'), analysis_rows, 'Noch keine Marktanalyse gespeichert.')}
<h3>Abgeschlossene virtuelle Trades · letzte 12</h3>
{table(('Coin', 'Kauf → Verkauf', 'Menge', 'Kaufkurs', 'Verkaufskurs', 'Ergebnis', 'Ausstieg'), trade_rows, 'Noch kein abgeschlossener Trade.')}
<h3>Letzte Ereignisse (seit Start dieser Sitzung)</h3>
{table(('Zeit', 'Coin', 'Ereignis', 'Details'), event_rows, 'Noch keine Ereignisse.')}
<div class="note">Paper Trading: keine echten Orders, keine Verbindung zu einem Kraken-Konto. Kurs = letzter Kraken-Handelspreis
zum Zeitpunkt der Kursprüfung (alle 5 Minuten). Ergebnisse enthalten die simulierten Kosten (Gebühr {percent(bot.GEBUEHR * 100, 1, False)},
Slippage {percent(bot.SLIPPAGE * 100, 1, False)} je Seite). Der Stop wird alle 5 Minuten gegen diesen Kurs geprüft und stündlich mit
{number(bot.ATR_TRAILING_MULTIPLIKATOR, 1)}×ATR nachgezogen; Kursbewegungen zwischen zwei Prüfungen sieht der Bot nicht.
Stop rot = unter Einstieg, grün = darüber. Tagesverlustlimit und Tageswechsel nach UTC.</div>
</div></body></html>'''


# ---------- browser tab and Drive file ----------

BROWSER_REFRESH_SECONDS = 15


def browser_page(page, refresh_seconds=BROWSER_REFRESH_SECONDS):
    """Same view for a normal browser tab: reloads itself and shows its age."""
    head = (f'<meta charset="utf-8"><meta http-equiv="refresh" content="{int(refresh_seconds)}">'
            '<meta name="viewport" content="width=device-width,initial-scale=1">')
    age = ('<p class="muted" id="kc-age"></p><script>(function(){'
           'var s=document.getElementById("kc-stand");if(!s)return;'
           'var e=document.getElementById("kc-age");'
           'var m=Math.floor((Date.now()-Number(s.dataset.updatedMs))/60000);if(!(m>=0))return;'
           f'e.textContent="Browseransicht lädt alle {int(refresh_seconds)} s neu · Stand ist "+m+" Min. alt";'
           'if(m>=Number(s.dataset.stallMin)&&s.dataset.quiet!=="1"){e.className="bad";'
           'e.textContent+=" – Bot läuft evtl. nicht mehr (Colab prüfen)";}})();</script>')
    return page.replace('<meta charset="utf-8">', head, 1).replace(
        '<div class="tiles">', age + '<div class="tiles">', 1)


WAITING_PAGE = ('<!doctype html><html lang="de"><head><meta charset="utf-8"></head>'
                '<body style="font:15px system-ui,sans-serif;padding:24px;background:#f5f8fc;color:#182338">'
                '<h2>Krypto-Bot · Paper-Trading-Cockpit</h2><p>Warte auf die erste Kursprüfung …</p>'
                '<div class="tiles"></div></body></html>')


def write_atomic(path, payload):
    temp = path.with_name(path.name + '.incomplete')
    temp.write_text(payload, encoding='utf-8')
    temp.replace(path)


class Cockpit:
    def __init__(self, display_enabled=True, latest_path=None):
        """`latest_path`: one fixed, self-refreshing HTML file (e.g. on Drive)
        that always shows the newest view."""
        self.display_enabled = display_enabled
        self.latest_path = Path(latest_path) if latest_path is not None else None
        self.latest_browser_page = browser_page(WAITING_PAGE)
        self.write_errors = 0
        self.handle = None

    def _show(self, page):
        if not self.display_enabled:
            return
        from IPython.display import HTML, display
        if self.handle is None:
            self.handle = display(HTML(page), display_id=True)
        else:
            self.handle.update(HTML(page))

    def show_placeholder(self):
        """Reserve the cockpit's place at the top of the cell output before the text log starts."""
        self._show(WAITING_PAGE)

    def update(self, lauf, stufe, ereignisse=(), now=None):
        data = snapshot(lauf, stufe, ereignisse, now)
        page = render_html(data)
        self.latest_browser_page = browser_page(page)   # served by the live link
        if self.latest_path is not None:
            try:
                self.latest_path.parent.mkdir(parents=True, exist_ok=True)
                write_atomic(self.latest_path, self.latest_browser_page)
            except OSError as exc:
                # A Drive hiccup must not switch the cockpit off; the next update writes again.
                self.write_errors += 1
                print('Cockpit-Datei nicht geschrieben:', self.latest_path.name, type(exc).__name__)
        self._show(page)
        return data


# ---------- read-only web server for the browser tab ----------

class _Handler(BaseHTTPRequestHandler):
    cockpit = None

    def do_GET(self):
        if self.path.split('?', 1)[0] not in ('/', '/index.html'):
            self.send_error(404)
            return
        payload = self.cockpit.latest_browser_page.encode('utf-8')
        self.send_response(200)
        self.send_header('Content-Type', 'text/html; charset=utf-8')
        self.send_header('Cache-Control', 'no-store')
        self.send_header('Content-Length', str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def log_message(self, *args):   # keep the notebook output clean
        pass


def serve_cockpit(cockpit, port=8766, attempts=20):
    """Serve the newest cockpit page over HTTP from a daemon thread.

    Read-only: only the page already rendered by Cockpit.update is returned;
    no market data, no account changes, no file system access."""
    handler = type('CockpitHandler', (_Handler,), dict(cockpit=cockpit))
    for candidate in range(port, port + attempts):
        try:
            server = ThreadingHTTPServer(('', candidate), handler)
        except OSError:
            continue   # e.g. a previous run of the cell still holds the port
        server.daemon_threads = True
        threading.Thread(target=server.serve_forever, daemon=True,
                         name='krypto-cockpit-server').start()
        return server
    raise OSError('Kein freier Port für die Cockpit-Browseransicht.')


def open_browser_view(cockpit, port=8766):
    """Start the server and, inside Colab, show a link that opens it in a browser tab."""
    server = serve_cockpit(cockpit, port)
    port = server.server_address[1]
    try:
        from google.colab import output
    except ImportError:
        print(f'Cockpit im Browser: http://localhost:{port}/')
        return server
    output.serve_kernel_port_as_window(port, path='/', anchor_text='Cockpit im Browser öffnen')
    print('Der Link funktioniert, solange diese Colab-Sitzung läuft; '
          f'die Seite lädt sich alle {BROWSER_REFRESH_SECONDS} s selbst neu.')
    return server


## Cockpit

Zuerst diese Zelle ausführen: Sie zeigt das Cockpit und den Browser-Link. Es wird von der Startzelle darunter laufend aktualisiert. Für eine Vollbildansicht ohne Code und Textlog im Ausgabemenü dieser Zelle „Copy link to fullscreen output“ wählen.

In [ ]:
import importlib
import sys

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import crypto_bot_v41 as bot
bot = importlib.reload(bot)
import crypto_cockpit_v41 as cockpit_ui
cockpit_ui = importlib.reload(cockpit_ui)

cockpit = cockpit_ui.Cockpit(display_enabled=True,
                             latest_path=bot.STANDARD_ORDNER / 'cockpit_latest.html')

# Die Ansicht gehört zu DIESER Zelle und wird aus der Startzelle
# darunter aktualisiert. Hier steht also nur das Cockpit, ohne Textlog:
# Ausgabemenü dieser Zelle -> "Copy link to fullscreen output".
cockpit.show_placeholder()

try:   # Browser-Tab zusätzlich; liefert nur die fertige Ansicht aus
    cockpit_ui.open_browser_view(cockpit)
except Exception as exc:
    print('Browser-Link nicht verfügbar:', type(exc).__name__,
          '– stattdessen MyDrive/Trading_Bot/cockpit_latest.html öffnen.')

## Start

Startet den Bot. Stoppen über ■ (Ausführung unterbrechen); das Paper-Konto bleibt in Google Drive gespeichert.

In [ ]:
# Die farbige Textausgabe erscheint wie in V4.0 hier, das Cockpit oben.
bot.start(cockpit=cockpit)